# EGM 722 Project Notebook: Greenspace Analysis (Draft)

## Introduction

Welcome to my project notebook! This will take you through the code required to perform analysis on the amount of Greenspace available in Northern Ireland. 

The code will consist of 4 main parts:
1. **Importing the necessary modules**
2. **Loading the data and setting the CRS**
3. **Carrying out the analysis**
4. **Mapping the results**

**Please Note:** This is currently only a draft - parts may be messy or incomplete!

## Part 1: Preparing Data

To start the project, we first need to import the necessary modules, followed by the data.

In [1]:
#Import the required modules
import os
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from cartopy.feature import ShapelyFeature
import cartopy.crs as ccrs

In [2]:
# Load and name the data layers
greenspace = gpd.read_file(os.path.abspath('data_files/Greenspace_Phase2_06052022.shp'))
lgd2014 = gpd.read_file(os.path.abspath('data_files/LGD2014.shp'))
dea2014 = gpd.read_file(os.path.abspath('data_files/dea2014.shp'))
dz2021 = gpd.read_file(os.path.abspath('data_files/DZ2021.shp'))
sdz2021 = gpd.read_file(os.path.abspath('data_files/SDZ2021.shp'))
settlements2015 = gpd.read_file(os.path.abspath('data_files/settlements-2015-above-500-threshold.shp'))


In [5]:
# View the first 5 rows of the greenspace data to check if it loaded in correctly
dz2021

,DZ2021_cd,DZ2021_nm,SDZ2021_cd,SDZ2021_nm,DEA2014_cd,DEA2014_nm,LGD2014_cd,LGD2014_nm,Area_ha,Perim_km,geometry
0,N20000001,Dunsilly_A1,N21000001,Dunsilly_A,N10000104,Dunsilly,N09000001,Antrim and Newtownabbey,2070.68,22.93,"POLYGON ((319580.264 396993.361, 319573.665 39..."
1,N20000002,Dunsilly_B1,N21000002,Dunsilly_B,N10000104,Dunsilly,N09000001,Antrim and Newtownabbey,1253.69,19.14,"POLYGON ((323772.447 393378.876, 323764.289 39..."
2,N20000003,Dunsilly_A2,N21000001,Dunsilly_A,N10000104,Dunsilly,N09000001,Antrim and Newtownabbey,792.91,15.50,"POLYGON ((305377.841 395055.242, 305378.841 39..."
3,N20000004,Dunsilly_A3,N21000001,Dunsilly_A,N10000104,Dunsilly,N09000001,Antrim and Newtownabbey,1100.52,17.90,"POLYGON ((311500.959 395028.333, 311510.728 39..."
4,N20000005,Dunsilly_B2,N21000002,Dunsilly_B,N10000104,Dunsilly,N09000001,Antrim and Newtownabbey,1740.82,21.93,"POLYGON ((323772.447 393378.876, 323784.201 39..."
...,...,...,...,...,...,...,...,...,...,...,...
3775,N20003776,Ards_Peninsula_N2,N21000850,Ards_Peninsula_N,N10001101,Ards Peninsula,N09000011,Ards and North Down,30.72,3.00,"POLYGON ((359752.134 351035.673, 359759.06 351..."
3776,N20003777,Ards_Peninsula_N3,N21000850,Ards_Peninsula_N,N10001101,Ards Peninsula,N09000011,Ards and North Down,7.41,1.21,"POLYGON ((359710.93 351090.923, 359759.06 3510..."
3777,N20003778,Ards_Peninsula_N4,N21000850,Ards_Peninsula_N,N10001101,Ards Peninsula,N09000011,Ards and North Down,18.28,3.06,"POLYGON ((359863.411 350571.537, 359828.275 35..."
3778,N20003779,Ards_Peninsula_N5,N21000850,Ards_Peninsula_N,N10001101,Ards Peninsula,N09000011,Ards and North Down,13.15,2.40,"POLYGON ((359671.758 349967.45, 359669.807 349..."


Now that the data is loaded and looks okay, we will check the crs of each layer, and make any necessary ammendments

In [4]:
# Check the crs of each layer
layer_crs = {'greenspace_crs': [greenspace.crs], 'lgd2014_crs': [lgd2014.crs], 'dz2021_crs': [dz2021.crs], 'sdz2021_crs': [sdz2021.crs], 'settlements2015_crs': [settlements2015.crs]}
crs_table = pd.DataFrame(data=layer_crs)
crs_table

,greenspace_crs,lgd2014_crs,dz2021_crs,sdz2021_crs,settlements2015_crs
0,EPSG:29902,EPSG:29902,EPSG:29902,EPSG:29902,EPSG:29902


In [6]:
ni_utm = ccrs.UTM(29) # create a Universal Transverse Mercator reference system to transform our data.

In [7]:
ccrs.CRS(greenspace.crs) # create a cartopy CRS representation of the CRS associated with the outline dataset


<Projected CRS: EPSG:29902>
Name: TM65 / Irish Grid
Axis Info [cartesian]:
- E[east]: Easting (metre)
- N[north]: Northing (metre)
Area of Use:
- name: Ireland - onshore.
- bounds: (-10.56, 51.39, -5.93, 55.43)
Coordinate Operation:
- name: Irish Grid
- method: Transverse Mercator
Datum: TM65
- Ellipsoid: Airy Modified 1849
- Prime Meridian: Greenwich

You may have noticed that the greenspace polygon has multiple 'Polgon Z' and 'Multipolygon Z' Types we want rid of these.

In [8]:
greenspace_sp=greenspace.explode(column=None)

## Part 2: Greenspace Analysis

This next part of the notebook will demonstrate some analysis using the greenspace data and boundaries provided.

Using the data, we can find:
1. What areas have the highest/lowest coverage of greenspace: at SDZ level and DEA, and per 1000 people.
2. What areas (SDZ) are closest or within a certain distance of greenspaces
3. Bonus: How many parcels are available within each DEA that are available for greenspace? These must be over a certain size, be a natural landscape, within distance of roads, within distance of houses and not already within a greenspace. Which DEAs have the most of these?

So, lets begin...


### Calculating coverage

Before we can calculate what area of greenspace is found within each SDZ, LGD, DEA and so on. It would be useful to calculate the area of each polygon to a standard unit.
This is where our first fucntion will be introduced, one that calculates the area of each polygon, in Km2 and adds it on as a new column to the table.

In [9]:
def area_calc(layer, col_name):
    """Caluclates the area of each polygon in the layer and returns a total.
    
    Parameters 
    layer : input polygon layer
    col_name : name of the area column

    Returns : sum_area_SQKM
        Prints the total area of the polygons in squared kilometers
    """

    layer[col_name] = layer['geometry'].area/1e6

    sum_area_SQKM = layer[col_name].sum()
    print(f'The total area is {sum_area_SQKM:.2f} kilometers squared')

In [10]:
#test area_calc function on greenspace layer
area_calc(greenspace_sp, 'greenspace_area_SQKM')

The total area is 875.94 kilometers squared


In [11]:
#test area_calc function column on greenspace layer
greenspace_sp

,SourceID,GUID,Name,Source,Category,Type,PaidAccess,Area_Ha,Verified,ShowOnMap,ORNI_ID,DataAdded,SiteCreate,geometry,greenspace_area_SQKM
0,NaN,68d2df20-1512-4218-a759-1000732e93b0,Conservation Volunteers NI,LPS and Outscape,Amenity Greenspace,Community Garden,No,0.668234,Approximated,Yes,51,2022-09-21,Pre 2023,"POLYGON Z ((335088.596 367463.239 0, 334995.26...",0.006682
1,NaN,7b20f220-682e-4566-a4fe-2513cc65ed6e,Lough Shore Park,Antrim and Newtownabbey Borough Council,Parks and Gardens,Public Park,No,6.598284,Approximated,Yes,61,2022-09-21,Pre 2023,"POLYGON Z ((313511.357 386601.644 0, 313465.51...",0.065983
2,NaN,8a4b8a31-eba9-4e7a-b8d5-cea0724d848d,Ardmore Recreation Centre,"Armagh City, Banbridge and Craigavon Borough C...",Amenity Greenspace,Playing Fields,No,2.969549,Approximated,Yes,67,2022-09-21,Pre 2023,"POLYGON Z ((289345.456 344248.549 0, 289352.83...",0.029695
3,NaN,36f2a040-649c-4eb5-9174-7b25c083f489,Clare Glen - Phase 3 - Link Footpath/River Wal...,"Armagh City, Banbridge and Craigavon Borough C...",Woodland,Woodland,No,3.396109,Approximated,Yes,69,2022-09-21,Pre 2023,"POLYGON Z ((303292.783 345625.097 0, 303291.38...",0.033961
4,NaN,2f586e46-b9ee-4d13-83b1-9b254fa3a698,"Folly Glen, Armagh","Armagh City, Banbridge and Craigavon Borough C...",Woodland,Woodland,No,7.849400,Approximated,Yes,70,2022-09-21,Pre 2023,"POLYGON Z ((288701.465 343901.121 0, 288688.70...",0.002338
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1487,NaN,NaN,Twinbrook Wildlife Park,Belfast City Council,Amenity Greenspace,Public Open Space,NaN,1.161658,NaN,Yes,0,1899-12-30,NaN,"POLYGON Z ((328479.994 368987.968 0, 328473.32...",0.010994
1487,NaN,NaN,Twinbrook Wildlife Park,Belfast City Council,Amenity Greenspace,Public Open Space,NaN,1.161658,NaN,Yes,0,1899-12-30,NaN,"POLYGON Z ((328487.187 369000.669 0, 328484.11...",0.000622
1488,NaN,NaN,Drumcoo Playing Fields,Mid Ulster District Council,Amenity Greenspace,Playing Fields,NaN,3.022465,NaN,Yes,0,1899-12-30,Pre 2023,"POLYGON Z ((279812.935 363921.041 0, 279810.78...",0.030225
1489,NaN,NaN,Battery Harbour Park,Mid Ulster District Council,Amentiy Greenspace,Public Open Space,NaN,1.769502,NaN,Yes,0,1899-12-30,NaN,"POLYGON Z ((296502.851 377054.243 0, 296514.15...",0.017695


In [12]:
help(area_calc)

Help on function area_calc in module __main__:

area_calc(layer, col_name)
    Caluclates the area of each polygon in the layer and returns a total.

    Parameters
    layer : input polygon layer
    col_name : name of the area column

    Returns : sum_area_SQKM
        Prints the total area of the polygons in squared kilometers



In [13]:
#add a area SQKM column to the remainder of the datasets and find their total area
area_calc(dz2021, 'dz_area')
area_calc(sdz2021, 'sdz_area')
area_calc(dea2014, 'dea_area')
area_calc(lgd2014, 'lgd_area')

The total area is 13628.31 kilometers squared
The total area is 13628.31 kilometers squared
The total area is 14315.28 kilometers squared
The total area is 14315.28 kilometers squared


Now we are ready to start analysing.


### Trying a new method: clipping!

In [14]:
def coverage_calc(layer, clipping_feature, orig_area):
    """Clips the selected greensapce layer to the selected dataset, creates a grouped gdf based on the individual features, which is then joined to the original selected dataset to calculate the % coverage

    Parameters: 
    layer - the selected layer or dataset to be clipped to
    clipping feature - the column name of the clipping dataset, must be a unique
    oirg_area - the calculated area column in the clipping dataset

    Returns:
    Output table - joined table of the original layer with information on its total coverage of greenspace
    """
    if layer[clipping_feature].is_unique == False:
        raise Exception('Clipping features must be unique.')
    
    clipped = []
    for selected_areas in layer[clipping_feature]:
        tmp_clip = gpd.clip(greenspace_sp, layer[layer[clipping_feature] == selected_areas])
        tmp_clip['greenspace_area_SQKM'] = tmp_clip['geometry'].area/1e6
        tmp_clip[clipping_feature] = selected_areas
    
        clipped.append(tmp_clip)

    clipped_gdf = gpd.GeoDataFrame(pd.concat(clipped, ignore_index=True))
    
    grouped_gdf = pd.DataFrame(index=layer[clipping_feature]) # Creates a grouped dataframe series that sums the total area of greenspace in the clipped layer
    grouped_gdf['greenspace_coverage_SQKM'] = clipped_gdf.groupby([clipping_feature])['greenspace_area_SQKM'].sum()

    output_table = pd.merge(layer, grouped_gdf, on = clipping_feature) # Joins the grouped data frame to the original dataset, and calculated the % coverage
    output_table['pc_coverage'] = output_table['greenspace_coverage_SQKM'] / output_table[orig_area] * 100

    return output_table

In [15]:
sdz_coverage = coverage_calc(sdz2021, 'SDZ2021_nm', 'sdz_area')

In [16]:
sdz_coverage.groupby(['SDZ2021_nm'])['pc_coverage'].sum().sort_values(ascending=False)

SDZ2021_nm
Castle_D            73.449311
Black_Mountain_D    55.353477
Macedon_D           47.536964
Titanic_C           44.800130
The_Moor_A          44.588771
                      ...    
Titanic_L            0.000000
Titanic_S            0.000000
Titanic_R            0.000000
Titanic_P            0.000000
Ards_Peninsula_B     0.000000
Name: pc_coverage, Length: 850, dtype: float64

In [17]:
lgd_coverage = coverage_calc(lgd2014, 'LGDNAME', 'lgd_area')

In [18]:
lgd_coverage

,OBJECTID,LGDNAME,LGDCode,popn,Shape_Leng,Shape_Area,geometry,lgd_area,greenspace_coverage_SQKM,pc_coverage
0,1.0,Antrim and Newtownabbey,N09000001,141428,165113.558771,7.278895e+08,"POLYGON ((319598.922 397397.797, 319595.68 397...",727.889478,12.425218,1.707020
1,2.0,Armagh City Banbridge and Craigavon,N09000002,209276,242967.696647,1.435980e+09,"POLYGON ((311885.478 366349.627, 311904.219 36...",1435.980451,29.361744,2.044718
2,3.0,Belfast,N09000003,337885,65905.220983,1.377021e+08,"POLYGON ((334071.541 379980.088, 334096.675 37...",137.702072,20.517916,14.900223
3,4.0,Causeway Coast and Glens,N09000004,143541,381602.781803,1.983852e+09,"MULTIPOLYGON (((297026.984 446000.593, 297033....",1983.851819,198.972460,10.029603
4,5.0,Derry City and Strabane,N09000005,150134,271234.988347,1.248976e+09,"POLYGON ((247533.964 424601.021, 247536.214 42...",1248.976336,76.646643,6.136757
5,6.0,Fermanagh and Omagh,N09000006,115794,426211.823548,3.005966e+09,"POLYGON ((266554.895 392093.167, 266557.237 39...",3005.965778,286.400462,9.527735
6,7.0,Lisburn and Castlereagh,N09000007,142587,167365.431586,5.103119e+08,"POLYGON ((343158.45 376583.005, 343162.985 376...",510.311902,5.642238,1.105645
7,8.0,Mid and East Antrim,N09000008,137427,234832.258324,1.061065e+09,"POLYGON ((329896.206 424716.479, 329900.103 42...",1061.065214,30.279666,2.853705
8,9.0,Mid Ulster,N09000009,145391,318235.783575,1.955057e+09,"POLYGON ((294150.193 412224.918, 294156.626 41...",1955.056614,73.207589,3.744525
9,10.0,Newry Mourne and Down,N09000010,178794,380041.040234,1.681856e+09,"POLYGON ((341276.618 362805.051, 341283.866 36...",1681.856446,117.673445,6.996640


### Finding Distances

For the next part of the analysis, we are going to demonstrate how python can be used to calculate distances between greenspaces and our areas. This will consist of: 

a) A buffer analysis, creating a function that allows you to easily calculate the number of greenspaces within a specified distance of your chosen location.
b) An interactive map, that shows the distance to the nearest greenspace for each DZ in the country.

In [19]:
settlements2015

,Code,Name,Band,DT_20MIN,DT_30MIN,UR_ex,OccHH_ex,UR_ap,OccHH_ap,geometry
0,N11000199,DUNNAMORE,H,Y,Y,119.0,35.0,119.0,35.0,"MULTIPOLYGON (((268852.639 381005.968, 268889...."
1,N11000015,BALLYBARNES,H,Y,Y,242.0,101.0,243.0,102.0,"POLYGON ((345603.666 375407.884, 345605.848 37..."
2,N11000121,LOUGHGUILE,H,N,Y,396.0,128.0,396.0,128.0,"POLYGON ((308240.408 425324.409, 308241.835 42..."
3,N11000265,BRYANSFORD,H,N,Y,306.0,114.0,309.0,115.0,"POLYGON ((334307.413 333016.05, 334309.718 333..."
4,N11000636,SION MILLS,G,Y,Y,1903.0,769.0,1907.0,770.0,"POLYGON ((233835.1 392685.125, 233833.438 3926..."
...,...,...,...,...,...,...,...,...,...,...
522,N11000457,UPPER BALLINDERRY,H,Y,Y,226.0,95.0,226.0,95.0,"POLYGON ((316438.311 367603.169, 316434.283 36..."
523,N11000153,CARRICKFERGUS,C,Y,Y,27903.0,11536.0,27998.0,11562.0,"POLYGON ((339592.352 389235.293, 339607.19 389..."
524,N11000443,LURGANVILLE,H,Y,Y,87.0,32.0,87.0,32.0,"POLYGON ((317538.491 359477.921, 317540.763 35..."
525,N11000151,BELFAST CITY,A,Y,Y,280211.0,120341.0,280138.0,120285.0,"MULTIPOLYGON (((332185.45 381234.249, 332185.4..."


In [18]:
#First see if any centroids do not fit within the settlement area
settlements2015.loc[~settlements2015['geometry'].contains(settlements2015.centroid)]

,Code,Name,Band,DT_20MIN,DT_30MIN,UR_ex,OccHH_ex,UR_ap,OccHH_ap,geometry
23,N11000085,MILLTOWN (BENBURB),H,Y,Y,108.0,39.0,108.0,39.0,"MULTIPOLYGON (((280562.585 351306.808, 280580...."
30,N11000023,BALLYHALBERT,G,N,N,1026.0,407.0,1040.0,414.0,"POLYGON ((363382.554 364070.777, 363389.608 36..."
34,N11000118,DUNAGHY,H,Y,Y,391.0,149.0,391.0,149.0,"MULTIPOLYGON (((297252.61 426241.837, 297314.1..."
38,N11000008,MONEYGLASS,H,N,Y,103.0,38.0,103.0,38.0,"POLYGON ((301694.028 393314.942, 301700.362 39..."
42,N11000402,DRUMSURN,H,Y,Y,459.0,161.0,452.0,159.0,"MULTIPOLYGON (((272069.047 417384.408, 272082...."
...,...,...,...,...,...,...,...,...,...,...
487,N11000561,ROUGHFORT,H,Y,Y,215.0,86.0,215.0,86.0,"MULTIPOLYGON (((327013.931 384022.779, 327017...."
494,N11000572,ORLOCK,H,Y,Y,126.0,50.0,128.0,51.0,"POLYGON ((355753.587 383596.517, 355754.888 38..."
499,N11000441,LOWER BROOMHEDGE,H,Y,Y,239.0,80.0,239.0,80.0,"POLYGON ((321293.411 362730.793, 321309.277 36..."
506,N11000422,DRUMBEG,H,Y,Y,813.0,321.0,817.0,323.0,"MULTIPOLYGON (((330603.615 367044.403, 330603...."


In [89]:
#Now create a copy of the settlments dataset, using a representative point (rather than centroids) 
settlements_pt = settlements2015.copy()
settlements_pt['geometry'] = settlements2015.representative_point()


In [39]:
#Select and display the chosen area to buffer
selected_area = settlements_pt.loc[settlements_pt['Name'] == 'BELFAST CITY']

selected_area

,Code,Name,Band,DT_20MIN,DT_30MIN,UR_ex,OccHH_ex,UR_ap,OccHH_ap,geometry
525,N11000151,BELFAST CITY,A,Y,Y,280211.0,120341.0,280138.0,120285.0,POINT (337676.36 374763.54)


In [90]:
#Create the buffer, in this example we will create a 1km buffer around Belfast
pt_buffer = selected_area.buffer(5000)

In [83]:
available_greenspace = greenspace.loc[greenspace['geometry'].within(pt_buffer.iloc[0])] #selects greenspaces that are within the buffer (first row of data using iloc)

In [82]:
available_greenspace.groupby(['Category'])['Name'].count()

Category
Amenity Greenspace    20
Bluespace              1
Nature Reserve         1
Parks and Gardens     20
Woodland               1
Name: Name, dtype: int64

Now let's write this as a function!

In [20]:
def buffer_analysis(layer, name_col, area, buffer_dist):
    """Counts the number of Greenspaces that are within a specified distance of your chosen feature

    Parameters:
    layer - chosen layer to measure distances from
    name_col - column name containing the reference name of chosesn feature
    area - name of chosen feature e.g. BELFAST CITY note: case sensitive
    buffer distance - selected buffer distance (in metres)
    
    Returns:
    A list count of the greenspaces within specified distance, grouped by category

    """

    layer_pt = layer.copy()
    layer_pt['geometry'] = layer.representative_point()

    selected_area = layer_pt.loc[layer_pt[name_col] == area]

    pt_buffer = selected_area.buffer(buffer_dist)

    available_greenspace = greenspace.loc[greenspace['geometry'].within(pt_buffer.iloc[0])]

    return available_greenspace.groupby(['Category'])['Name'].count()


In [21]:
buffer_analysis(settlements2015, 'Name', 'DONAGHCLONEY', 10000)

Category
Amenity Greenspace    17
Parks and Gardens      8
Woodland               3
Name: Name, dtype: int64

Next Part: Make an interactive map showing the number of greenspaces and distance to the nearest greenspace for each area

In [23]:
#for each dz centroid, find the closest greenspace
#report the distance in km, and the name of the greenspace
for ind, row in dz2021.iterrows():
    pt = row['geometry'].centroid
    distances = greenspace.distance(pt)

    min_ind = distances.argmin()
    min_dist = distances.min()

    dz2021.loc[ind, 'Nearest_gSpace'] = greenspace.loc[min_ind].Name
    dz2021.loc[ind, 'Distance_km'] = min_dist/1000

dz2021.Distance_km = dz2021.Distance_km.round(2)

In [64]:
#Count the number of greenspaces per dz
joined = gpd.sjoin(dz2021, greenspace, how='inner', lsuffix='left', rsuffix='right')

In [65]:
num_gspace= joined.groupby('DZ2021_nm')
num_gspace['DZ2021_nm'].count()

DZ2021_nm
Airport_A3        1
Airport_B1        2
Airport_B3        1
Airport_C1        1
Airport_C2        1
                 ..
West_Tyrone_F3    1
West_Tyrone_G1    1
West_Tyrone_G3    1
West_Tyrone_G4    1
West_Tyrone_G6    1
Name: DZ2021_nm, Length: 1665, dtype: int64

In [66]:
num_space_df = pd.DataFrame(index=dz2021['DZ2021_nm'])
num_space_df['Num_gSpaces'] = num_gspace['DZ2021_nm'].count()
to_join=num_space_df.fillna(0)

In [61]:
merged = dz2021.merge(to_join, left_on='DZ2021_nm', right_on='DZ2021_nm')

In [67]:
merged

,DZ2021_cd,DZ2021_nm,SDZ2021_cd,SDZ2021_nm,DEA2014_cd,DEA2014_nm,LGD2014_cd,LGD2014_nm,Area_ha,Perim_km,geometry,dz_area,Nearest_gSpace,Distance_km,Num_gSpaces
0,N20000001,Dunsilly_A1,N21000001,Dunsilly_A,N10000104,Dunsilly,N09000001,Antrim and Newtownabbey,2070.68,22.93,"POLYGON ((319580.264 396993.361, 319573.665 39...",20.706758,Tardree,1.01,1.0
1,N20000002,Dunsilly_B1,N21000002,Dunsilly_B,N10000104,Dunsilly,N09000001,Antrim and Newtownabbey,1253.69,19.14,"POLYGON ((323772.447 393378.876, 323764.289 39...",12.536913,NaN,2.88,0.0
2,N20000003,Dunsilly_A2,N21000001,Dunsilly_A,N10000104,Dunsilly,N09000001,Antrim and Newtownabbey,792.91,15.50,"POLYGON ((305377.841 395055.242, 305378.841 39...",7.929093,NaN,3.60,0.0
3,N20000004,Dunsilly_A3,N21000001,Dunsilly_A,N10000104,Dunsilly,N09000001,Antrim and Newtownabbey,1100.52,17.90,"POLYGON ((311500.959 395028.333, 311510.728 39...",11.005218,NaN,3.95,0.0
4,N20000005,Dunsilly_B2,N21000002,Dunsilly_B,N10000104,Dunsilly,N09000001,Antrim and Newtownabbey,1740.82,21.93,"POLYGON ((323772.447 393378.876, 323784.201 39...",17.408166,Tardree,2.57,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3775,N20003776,Ards_Peninsula_N2,N21000850,Ards_Peninsula_N,N10001101,Ards Peninsula,N09000011,Ards and North Down,30.72,3.00,"POLYGON ((359752.134 351035.673, 359759.06 351...",0.307189,Cloughy Road Playing Fields,0.20,1.0
3776,N20003777,Ards_Peninsula_N3,N21000850,Ards_Peninsula_N,N10001101,Ards Peninsula,N09000011,Ards and North Down,7.41,1.21,"POLYGON ((359710.93 351090.923, 359759.06 3510...",0.074098,Ann Street,0.26,0.0
3777,N20003778,Ards_Peninsula_N4,N21000850,Ards_Peninsula_N,N10001101,Ards Peninsula,N09000011,Ards and North Down,18.28,3.06,"POLYGON ((359863.411 350571.537, 359828.275 35...",0.182796,Ann Street,0.39,0.0
3778,N20003779,Ards_Peninsula_N5,N21000850,Ards_Peninsula_N,N10001101,Ards Peninsula,N09000011,Ards and North Down,13.15,2.40,"POLYGON ((359671.758 349967.45, 359669.807 349...",0.131543,Nugents Wood,0.71,0.0
